# Data Skills&Tricks

# 1pandas因子编写进阶操作

## 1.1透视表（pivot table） 

数据汇总和分析工具   根据一个或多个列中的数据创建多维数据汇总表格
- 多维汇总
- 灵活 回测分析收益系列
- 聚合函数 

In [21]:
#透视表
import pandas as pd
data = {
    'Salesperson':['Alice','Bob','Charlie','Alice','Bob','Charile'],
    'Category':['Electronics','Electronics','Electronics','Clothing','Clothing','Clothing'],
    'SalesAmount':[500,750,300,200,300,150]
}
df = pd.DataFrame(data)
df

,Salesperson,Category,SalesAmount
0,Alice,Electronics,500
1,Bob,Electronics,750
2,Charlie,Electronics,300
3,Alice,Clothing,200
4,Bob,Clothing,300
5,Charile,Clothing,150


In [22]:
#创建透视表，指定多个聚合函数
pivot_table = pd.pivot_table(df,values= 'SalesAmount',index='Salesperson',columns='Category',aggfunc=['sum','mean'])
pivot_table

sum                 mean            
Category    Clothing Electronics Clothing Electronics
Salesperson                                          
Alice          200.0       500.0    200.0       500.0
Bob            300.0       750.0    300.0       750.0
Charile        150.0         NaN    150.0         NaN
Charlie          NaN       300.0      NaN       300.0

## 1.2日期时间·数据操作

- 灵活的时间索引  支持 重采样 时间差 窗口操作         
- 例如 给了日频的 结果人家要一周的财报 就可以重采样 在重采样范围里做聚合操作 
- 例如10：31：42    10-31-42  或者10/31/42 转换字符串形式
- 例如没有补齐的 09/35/20   有时候吧09的0去掉  就不能用字符串了

- 时间序列操作 重采样 滑动窗口计算 移动平均值计算 时间差分析
- 日期 时间格式化和解析 s


In [3]:
import pandas as pd
data = {'Date': ['2023-01-01', '2023-02-15', '2023-03-30'],
        'Value': [100, 200, 150]}
df = pd.DataFrame(data)
df

,Date,Value
0,2023-01-01,100
1,2023-02-15,200
2,2023-03-30,150


In [4]:
# 将日期列从字符串转换为日期对象
df['Date'] = pd.to_datetime(df['Date'])
df

,Date,Value
0,2023-01-01,100
1,2023-02-15,200
2,2023-03-30,150


In [3]:
type(df['Date'].values[0])

numpy.datetime64

In [5]:
df['Formatted_Date'] = df['Date'].dt.strftime('%Y/%m/%d')
df

,Date,Value,Formatted_Date
0,2023-01-01,100,2023/01/01
1,2023-02-15,200,2023/02/15
2,2023-03-30,150,2023/03/30


In [6]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df

,Date,Value,Formatted_Date,Year,Month,Day
0,2023-01-01,100,2023/01/01,2023,1,1
1,2023-02-15,200,2023/02/15,2023,2,15
2,2023-03-30,150,2023/03/30,2023,3,30


In [8]:
# 计算日期差
df['Days_Since_Start'] = (df['Date'] - df['Date'].min()).dt.days
df

,Date,Value,Formatted_Date,Year,Month,Day,Days_Since_Start
0,2023-01-01,100,2023/01/01,2023,1,1,0
1,2023-02-15,200,2023/02/15,2023,2,15,45
2,2023-03-30,150,2023/03/30,2023,3,30,88


In [10]:
# 判断是否为工作日
df['Is_Workday'] = df['Date'].dt.weekday < 5
df

,Date,Value,Formatted_Date,Year,Month,Day,Days_Since_Start,Tomorrow,Is_Workday
0,2023-01-01,100,2023/01/01,2023,1,1,0,2023-01-02,False
1,2023-02-15,200,2023/02/15,2023,2,15,45,2023-02-16,True
2,2023-03-30,150,2023/03/30,2023,3,30,88,2023-03-31,True


In [9]:
# 增加一天
df['Tomorrow'] = df['Date'] + pd.Timedelta(days=1)
df

,Date,Value,Formatted_Date,Year,Month,Day,Days_Since_Start,Tomorrow
0,2023-01-01,100,2023/01/01,2023,1,1,0,2023-01-02
1,2023-02-15,200,2023/02/15,2023,2,15,45,2023-02-16
2,2023-03-30,150,2023/03/30,2023,3,30,88,2023-03-31


## 1.3Rolling函数


- 执行滚动窗口计算
- 数据帧
- 任何一维数据都可以用

In [12]:
import pandas as pd
import numpy as np
import glob
import os
# rolling函数简介：可以在滑动窗口上执行各类操作  


data = pd.Series([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
data

0     1
1     2
2     3
3     4
4     5
5     6
6     7
7     8
8     9
9    10
dtype: int64

In [15]:
# 使用 rolling 函数计算滑动窗口大小为3的平均值
rolling_mean = data.rolling(window=3).mean() 
rolling_mean

0    NaN
1    NaN
2    2.0
3    3.0
4    4.0
5    5.0
6    6.0
7    7.0
8    8.0
9    9.0
dtype: float64

- DataFrame.rolling(window,min_periods=None,center=False,axis=0)
- window:指定窗口的大小，可以是一个整数，或者一个时间周期
- min_periods 计算统计量之前 窗口必须有的非空数据点数 默认为窗口大小 （例如上例中前两个就不行）
- center 窗口是否应该居中在数据点上 默认为窗口右对齐 False
- axis 指定计算的值 0行 1列

## 1.5Group 函数
- 数据分组 分组统计

In [4]:
import pandas as pd
# 创建示例数据帧
data = {
    'Category': ['A', 'B', 'A', 'B', 'A', 'B'],
    'Value': [10, 20, 15, 25, 30, 35]
}
df = pd.DataFrame(data)
df

,Category,Value
0,A,10
1,B,20
2,A,15
3,B,25
4,A,30
5,B,35


In [6]:
# 按 'Category' 列分组，计算每组的平均值
grouped = df.groupby('Category')
mean_values = grouped['Value'].mean()
print(mean_values)

Category
A    18.333333
B    26.666667
Name: Value, dtype: float64


In [7]:
# 自定义聚合函数
def custom_agg(column):
    return column.max() - column.min()

agg_result = grouped['Value'].agg(custom_agg)
print(agg_result)

Category
A    20
B    15
Name: Value, dtype: int64


Group 函数总结
DadaFrame.groupby(by=None,axis = 0,sort = True.squeeze = False)
- by指定用于分组的列名 或列名的列表
- axis 0行 1列
- sort 布尔值 是否对分组键进行排序
- squeeze 布尔值 True的话返回对象是Series 不是DataFrame

## 1.6 Apply函数 
- 用于数据帧 或者数据系列上应用自定义函数 
- 用于对整列 整行或数据帧中的每个元素执行操作

In [21]:
import pandas as pd

data = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Charlie', 'David'],
    'Age': [25, 30, 35, 40],
    'Salary': [50000, 60000, 70000, 80000]
})
data

,Name,Age,Salary
0,Alice,25,50000
1,Bob,30,60000
2,Charlie,35,70000
3,David,40,80000


In [22]:
# 定义一个自定义函数，将年龄转换为年龄段
def convert_age(age):
    if age < 30:
        return 'Young'
    else:
        return 'Old'


# 使用 apply 函数将年龄转换为年龄段
data['Age Group'] = data['Age'].apply(convert_age)
print(data)

      Name  Age  Salary Age Group
0    Alice   25   50000     Young
1      Bob   30   60000       Old
2  Charlie   35   70000       Old
3    David   40   80000       Old


In [23]:
def convert_salary(salary):
    if salary <= 60000:
        return 'not_rich'
    else:
        return 'rich'
    
data['Salary_level'] = data['Salary'].apply(convert_salary)
print(data)

      Name  Age  Salary Age Group Salary_level
0    Alice   25   50000     Young     not_rich
1      Bob   30   60000       Old     not_rich
2  Charlie   35   70000       Old         rich
3    David   40   80000       Old         rich


In [24]:
# 在 pandas 的 apply 函数中，lambda 函数通常用于对数据进行元素级别的操作。lambda 函数是一种匿名函数，
# 它允许定义一个简单的函数，而无需显式地定义函数的名称。

df = data
# 使用Lambda函数和额外参数对列应用自定义操作
extra_param = 10
df['Age_with_param'] = df['Age'].apply(lambda x, param: x + param, args=(extra_param,))
print(df)


      Name  Age  Salary Age Group Salary_level  Age_with_param
0    Alice   25   50000     Young     not_rich              35
1      Bob   30   60000       Old     not_rich              40
2  Charlie   35   70000       Old         rich              45
3    David   40   80000       Old         rich              50


In [25]:
# 使用Lambda函数对多列应用自定义操作
df['Combined'] = df.apply(lambda row: f"{row['Age_with_param']} - {row['Age']}", axis=1)
print(df)

      Name  Age  Salary Age Group Salary_level  Age_with_param Combined
0    Alice   25   50000     Young     not_rich              35  35 - 25
1      Bob   30   60000       Old     not_rich              40  40 - 30
2  Charlie   35   70000       Old         rich              45  45 - 35
3    David   40   80000       Old         rich              50  50 - 40


# 2  数据文件读写拼接

In [32]:
# 读取多个数据文件拼接在一起
import pandas as pd
import numpy as np
import glob
import os
path = r'E:\OB\projects\DEMO\Knowledge\博一\宽客\python-for-quant\notes\quant\quant class\公开股票数据'
all_files = glob.glob(path + "/*.csv")   # 获取某个文件夹下所有数据文件的路径
all_files

['E:\\OB\\projects\\DEMO\\Knowledge\\博一\\宽客\\python-for-quant\\notes\\quant\\quant class\\公开股票数据\\finance-charts-apple.csv']

In [33]:
df = pd.read_csv(r'E:\OB\projects\DEMO\Knowledge\博一\宽客\python-for-quant\notes\quant\quant class\公开股票数据\finance-charts-apple.csv', index_col=None, header=0)
df

,Date,AAPL.Open,AAPL.High,AAPL.Low,AAPL.Close,AAPL.Volume,AAPL.Adjusted,dn,mavg,up,direction
0,2015-02-17,127.489998,128.880005,126.919998,127.830002,63152400,122.905254,106.741052,117.927667,129.114281,Increasing
1,2015-02-18,127.629997,128.779999,127.449997,128.720001,44891700,123.760965,107.842423,118.940333,130.038244,Increasing
2,2015-02-19,128.479996,129.029999,128.330002,128.449997,37362400,123.501363,108.894245,119.889167,130.884089,Decreasing
3,2015-02-20,128.619995,129.500000,128.050003,129.500000,48948400,124.510914,109.785449,120.763500,131.741551,Increasing
4,2015-02-23,130.020004,133.000000,129.660004,133.000000,70974100,127.876074,110.372516,121.720167,133.067817,Increasing
...,...,...,...,...,...,...,...,...,...,...,...
501,2017-02-10,132.460007,132.940002,132.050003,132.119995,20065500,132.119995,114.494004,124.498666,134.503328,Decreasing
502,2017-02-13,133.080002,133.820007,132.750000,133.289993,23035400,133.289993,114.820798,125.205166,135.589534,Increasing
503,2017-02-14,133.470001,135.089996,133.250000,135.020004,32815500,135.020004,115.175718,125.953499,136.731280,Increasing
504,2017-02-15,135.520004,136.270004,134.619995,135.509995,35501600,135.509995,115.545035,126.723499,137.901963,Decreasing


In [35]:
# 读取多个数据文件拼接在一起
import pandas as pd
import numpy as np
import glob
import os
path = r'E:\OB\projects\DEMO\Knowledge\博一\宽客\python-for-quant\notes\quant\quant class\公开股票数据'
all_files = glob.glob(path + "/*.csv")   # 获取某个文件夹下所有数据文件的路径
all_files
li = []
for filename in all_files:
    df = pd.read_csv(filename, index_col=None, header=0)
    li.append(df)
frame = pd.concat(li, axis=0, ignore_index=True)
frame

,Date,AAPL.Open,AAPL.High,AAPL.Low,AAPL.Close,AAPL.Volume,AAPL.Adjusted,dn,mavg,up,direction
0,2015-02-17,127.489998,128.880005,126.919998,127.830002,63152400,122.905254,106.741052,117.927667,129.114281,Increasing
1,2015-02-18,127.629997,128.779999,127.449997,128.720001,44891700,123.760965,107.842423,118.940333,130.038244,Increasing
2,2015-02-19,128.479996,129.029999,128.330002,128.449997,37362400,123.501363,108.894245,119.889167,130.884089,Decreasing
3,2015-02-20,128.619995,129.500000,128.050003,129.500000,48948400,124.510914,109.785449,120.763500,131.741551,Increasing
4,2015-02-23,130.020004,133.000000,129.660004,133.000000,70974100,127.876074,110.372516,121.720167,133.067817,Increasing
...,...,...,...,...,...,...,...,...,...,...,...
501,2017-02-10,132.460007,132.940002,132.050003,132.119995,20065500,132.119995,114.494004,124.498666,134.503328,Decreasing
502,2017-02-13,133.080002,133.820007,132.750000,133.289993,23035400,133.289993,114.820798,125.205166,135.589534,Increasing
503,2017-02-14,133.470001,135.089996,133.250000,135.020004,32815500,135.020004,115.175718,125.953499,136.731280,Increasing
504,2017-02-15,135.520004,136.270004,134.619995,135.509995,35501600,135.509995,115.545035,126.723499,137.901963,Decreasing
